In [2]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("data.csv")
df

,pos_x,pscale
0,0.000000,0.0000
1,0.161389,0.0248
2,0.322778,0.0496
3,0.484167,0.0744
4,0.645556,0.0992
5,0.806944,0.1240
6,0.968333,0.1488
7,1.129722,0.1736
8,1.291111,0.1984
9,1.452500,0.2232


In [23]:
import numpy as np
from sklearn.linear_model import LinearRegression

# Prepare data
x_pos = np.arange(len(df)).reshape(-1, 1)  # Now reshape the array, not the integer
y_pos = df['pos_x'].values                 # y values

# Create and train model
model_pos = LinearRegression()
model_pos.fit(x_pos, y_pos )
next_pos_x = model_pos.predict([[len(df)]])[0]

In [49]:
import pickle
# Create and train model
model_pos = LinearRegression()
model_pos.fit(x_pos, y_pos )
next_pos_x = model_pos.predict([[len(df)]])[0]

with open("mode_pos.pk1","wb") as f:
  pickle.dump(model_pos,f)

In [24]:
next_pos_x

np.float64(3.0663888812587974)

In [25]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

poly = PolynomialFeatures(degree=3)
x_pscale = poly.fit_transform(np.arange(len(df)).reshape(-1,1))
y_pscale = df['pscale'].values

model_pscale = LinearRegression()
model_pscale.fit(x_pscale, y_pscale)
next_pscale = model_pscale.predict(poly.transform([[len(df)]]))[0]

In [26]:
next_pscale

np.float64(0.4711999829178174)

In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from sklearn.preprocessing import MinMaxScaler

In [29]:
df.head()

,pos_x,pscale
0,0.000000,0.0000
1,0.161389,0.0248
2,0.322778,0.0496
3,0.484167,0.0744
4,0.645556,0.0992


In [32]:
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df)

scaled

array([[0.        , 0.        ],
       [0.05555556, 0.05555555],
       [0.11111111, 0.11111111],
       [0.16666666, 0.16666666],
       [0.22222222, 0.22222222],
       [0.27777778, 0.27777778],
       [0.33333333, 0.33333331],
       [0.3888889 , 0.38888887],
       [0.44444445, 0.44444444],
       [0.5       , 0.5       ],
       [0.55555555, 0.55555556],
       [0.6111111 , 0.61111113],
       [0.66666665, 0.66666662],
       [0.72222225, 0.72222225],
       [0.7777778 , 0.77777775],
       [0.83333335, 0.83333338],
       [0.8888889 , 0.88888887],
       [0.94444445, 0.94444444],
       [1.        , 1.        ]])

In [34]:
def create_sequences(data, seq_length):
  x, y = [], []
  for i in range(len(data) - seq_length):
    x.append(data[i:(i + seq_length)])
    y.append(data[i + seq_length])
  return np.array(x), np.array(y)

In [35]:
sequence_length = 3
x,y = create_sequences(scaled, sequence_length)

In [40]:
#buidling LSTM model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

#model
model = Sequential()
model.add(LSTM(50, activation= 'relu', input_shape=(x.shape[1], x.shape[2])))
model.add(Dense(2))

#compile
model.compile(optimizer = 'adam', loss ='mse')


In [42]:
#train the model
model.fit(x,y , epochs =200, verbose=1)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 0.4311
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - loss: 0.4247
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - loss: 0.4185
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 0.4123
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 0.4062
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.4003
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 0.3946
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.3890
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 0.3834
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - loss: 0.3780
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - loss: 0.3726
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.3672
Epoch 13/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 0.3618
Epoch 14/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 0.3565
Epoch 15/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 0.3511
Epoch 16/200
1/1 ━━━

In [43]:
last_seq = x[-1:]
predicted = model.predict(last_seq)

predicted_original_scale = scaler.inverse_transform(predicted)
print("Predicted next position:",predicted_original_scale[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 474ms/step
Predicted next position: [2.9743247  0.45784846]


In [44]:
df

,pos_x,pscale
0,0.000000,0.0000
1,0.161389,0.0248
2,0.322778,0.0496
3,0.484167,0.0744
4,0.645556,0.0992
5,0.806944,0.1240
6,0.968333,0.1488
7,1.129722,0.1736
8,1.291111,0.1984
9,1.452500,0.2232


In [48]:
# recursive predication
num_predictions = 5
predictions = []

last_seq = x[-1:]

for _ in range(num_predictions):
  predicted = model.predict(last_seq)
  predictions.append(predicted[0])
  predicted_reshaped = predicted.reshape(1,1,2)
  last_seq = np.concatenate((last_seq[:,1:,:], predicted_reshaped), axis=1)

predicted = scaler.inverse_transform(predictions)
print("Predictions:", predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
Predictions: [array([1.0238639, 1.0256462], dtype=float32), array([1.1047021, 1.1080929], dtype=float32), array([1.2013535, 1.2064271], dtype=float32), array([1.3243384, 1.3319473], dtype=float32), array([1.4736185, 1.4849658], dtype=float32)]
